# I. Khởi tạo SparkSession

In [1]:
import os
import sys

os.environ["JAVA_HOME"] = r"C:\Users\beste\Downloads\jdk-17.0.12_windows-x64_bin\jdk-17.0.12"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Nyc_Taxi_Trip_Duration") \
    .master("local[2]") \
    .getOrCreate()

sc = spark.sparkContext

print(sc)

<SparkContext master=local[2] appName=Nyc_Taxi_Trip_Duration>


# II. Đọc dữ liệu

Đọc dữ liệu từ file `train.csv`.

In [2]:
df = spark.read.csv(
    "data/train.csv",
    header=True,
    inferSchema=True
)

# III. Hiển thị dữ liệu

Hiển thị 5 dòng đầu tiên của tập dữ liệu.

In [3]:
df.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

# IV. Kiểm tra cấu trúc dữ liệu

In ra cấu trúc và kiểu dữ liệu của các thuộc tính.

In [4]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



# V. Kiểm tra kích thước dữ liệu

Đếm số dòng và số cột của tập dữ liệu.

In [5]:
print("Số dòng:", df.count())
print("Số cột:", len(df.columns))

Số dòng: 1458644
Số cột: 11


# VI. Hiển thị danh sách các thuộc tính

In ra tên tất cả các cột trong tập dữ liệu.

In [6]:
print(df.columns)

['id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration']


# VII. Mô tả dữ liệu
Các thuộc tính chính trong tập dữ liệu:

- `id`: Mã chuyến taxi.
- `vendor_id`: Mã nhà cung cấp dữ liệu.
- `pickup_datetime`: Thời gian đón khách.
- `dropoff_datetime`: Thời gian trả khách.
- `passenger_count`: Số lượng hành khách.
- `pickup_longitude`: Kinh độ điểm đón.
- `pickup_latitude`: Vĩ độ điểm đón.
- `dropoff_longitude`: Kinh độ điểm trả.
- `dropoff_latitude`: Vĩ độ điểm trả.
- `store_and_fwd_flag`: Trạng thái lưu và chuyển dữ liệu.
- `trip_duration`: Thời gian chuyến đi tính bằng giây.

# VIII. Kiểm tra và làm sạch dữ liệu

Trước khi phân tích luồng giao thông, cần kiểm tra chất lượng dữ liệu và loại bỏ các bản ghi không hợp lệ.

In [7]:
from pyspark.sql.functions import col, count, when

## VIII.1. Kiểm tra giá trị NULL

Kiểm tra số lượng giá trị NULL trong từng thuộc tính của tập dữ liệu.

In [8]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
| id|vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
|  0|        0|              0|               0|              0|               0|              0|                0|               0|                 0|            0|
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+



## VIII.2. Kiểm tra dữ liệu trùng lặp

Kiểm tra xem trong tập dữ liệu có các dòng bị trùng lặp hoàn toàn hay không.

In [9]:
tong_dong = df.count()
dong_khong_trung = df.dropDuplicates().count()

so_dong_trung = tong_dong - dong_khong_trung

print("Số dòng trùng lặp:", so_dong_trung)

Số dòng trùng lặp: 0


## VIII.3. Kiểm tra số lượng hành khách

Kiểm tra các giá trị xuất hiện trong thuộc tính `passenger_count`.

In [10]:
df.groupBy("passenger_count") \
    .count() \
    .orderBy("passenger_count") \
    .show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|              0|     60|
|              1|1033540|
|              2| 210318|
|              3|  59896|
|              4|  28404|
|              5|  78088|
|              6|  48333|
|              7|      3|
|              8|      1|
|              9|      1|
+---------------+-------+



Loại bỏ các chuyến đi có số lượng hành khách không hợp lệ. Chỉ giữ các chuyến có số lượng hành khách từ 1 đến 6.

In [11]:
df_clean = df.filter(
    (col("passenger_count") >= 1) &
    (col("passenger_count") <= 6)
)

print("Số dòng sau khi lọc hành khách:", df_clean.count())

Số dòng sau khi lọc hành khách: 1458579


In [12]:
df_clean

DataFrame[id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, pickup_longitude: double, pickup_latitude: double, dropoff_longitude: double, dropoff_latitude: double, store_and_fwd_flag: string, trip_duration: int]

## VIII.4. Kiểm tra thời gian chuyến đi

Thuộc tính `trip_duration` biểu diễn thời gian của chuyến taxi, đơn vị là giây.

Tiến hành thống kê để phát hiện các giá trị bất thường.

In [13]:
df_clean.select("trip_duration").describe().show()

+-------+-----------------+
|summary|    trip_duration|
+-------+-----------------+
|  count|          1458579|
|   mean|959.4638466617166|
| stddev|5237.072507533724|
|    min|                1|
|    max|          3526282|
+-------+-----------------+



Kiểm tra thời gian chuyến đi nhỏ nhất và lớn nhất.

In [14]:
from pyspark.sql.functions import min, max

df_clean.select(
    min("trip_duration").alias("Thời gian nhỏ nhất"),
    max("trip_duration").alias("Thời gian lớn nhất")
).show()

+------------------+------------------+
|Thời gian nhỏ nhất|Thời gian lớn nhất|
+------------------+------------------+
|                 1|           3526282|
+------------------+------------------+



Loại bỏ các chuyến đi có thời gian quá ngắn hoặc quá dài. Chỉ giữ các chuyến có thời gian từ 1 phút đến 6 giờ.

In [15]:
df_clean = df_clean.filter(
    (col("trip_duration") >= 60) &
    (col("trip_duration") <= 21600)
)

print("Số dòng sau khi lọc thời gian:", df_clean.count())

Số dòng sau khi lọc thời gian: 1447969


## VIII.5. Kiểm tra dữ liệu tọa độ

Kiểm tra giá trị nhỏ nhất và lớn nhất của tọa độ điểm đón và điểm trả.

Các tọa độ bất thường nằm quá xa khu vực New York sẽ được loại bỏ.

In [16]:
df_clean.select(
    min("pickup_longitude").alias("pickup_longitude_min"),
    max("pickup_longitude").alias("pickup_longitude_max"),
    min("pickup_latitude").alias("pickup_latitude_min"),
    max("pickup_latitude").alias("pickup_latitude_max")
).show()

+--------------------+--------------------+-------------------+-------------------+
|pickup_longitude_min|pickup_longitude_max|pickup_latitude_min|pickup_latitude_max|
+--------------------+--------------------+-------------------+-------------------+
| -121.93334197998047|  -61.33552932739258|  34.35969543457031|  51.88108444213867|
+--------------------+--------------------+-------------------+-------------------+



Kiểm tra tọa độ tại điểm trả khách.

In [17]:
df_clean.select(
    min("dropoff_longitude").alias("dropoff_longitude_min"),
    max("dropoff_longitude").alias("dropoff_longitude_max"),
    min("dropoff_latitude").alias("dropoff_latitude_min"),
    max("dropoff_latitude").alias("dropoff_latitude_max")
).show()

+---------------------+---------------------+--------------------+--------------------+
|dropoff_longitude_min|dropoff_longitude_max|dropoff_latitude_min|dropoff_latitude_max|
+---------------------+---------------------+--------------------+--------------------+
|  -121.93330383300781|   -61.33552932739258|    32.1811408996582|   43.92102813720703|
+---------------------+---------------------+--------------------+--------------------+



## VIII.6. Lọc tọa độ khu vực New York

Giới hạn dữ liệu trong vùng tọa độ nghiên cứu xung quanh thành phố New York.

Khoảng tọa độ được sử dụng:

- Latitude: từ 40.5 đến 41.0
- Longitude: từ -74.3 đến -73.6

In [18]:
df_clean = df_clean.filter(
    (col("pickup_latitude") >= 40.5) &
    (col("pickup_latitude") <= 41.0) &
    (col("pickup_longitude") >= -74.3) &
    (col("pickup_longitude") <= -73.6) &
    (col("dropoff_latitude") >= 40.5) &
    (col("dropoff_latitude") <= 41.0) &
    (col("dropoff_longitude") >= -74.3) &
    (col("dropoff_longitude") <= -73.6)
)

## VIII.7. Kiểm tra dữ liệu sau khi làm sạch

Kiểm tra số lượng dữ liệu trước và sau quá trình làm sạch.

In [19]:
print("Số dòng dữ liệu ban đầu:", df.count())
print("Số dòng sau khi làm sạch:", df_clean.count())

Số dòng dữ liệu ban đầu: 1458644
Số dòng sau khi làm sạch: 1447446


Hiển thị một số bản ghi sau khi làm sạch dữ liệu.

In [20]:
df_clean.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

## VIII.8. Kiểm tra thống kê dữ liệu sau khi làm sạch

In [21]:
df_clean.select(
    "passenger_count",
    "trip_duration",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
).describe().show()

+-------+------------------+-----------------+-------------------+------------------+--------------------+------------------+
|summary|   passenger_count|    trip_duration|   pickup_longitude|   pickup_latitude|   dropoff_longitude|  dropoff_latitude|
+-------+------------------+-----------------+-------------------+------------------+--------------------+------------------+
|  count|           1447446|          1447446|            1447446|           1447446|             1447446|           1447446|
|   mean|1.6654417505039911|841.3971940922148| -73.97359063836394|40.750995966758616|  -73.97353220714999| 40.75184154740706|
| stddev|1.3147691849102345|660.9237576702219|0.03778996530472351|0.0278750702540525|0.035020087375957076|0.0319925748951292|
|    min|                 1|               60| -74.28901672363281| 40.50629425048828|   -74.2929916381836| 40.50859832763672|
|    max|                 6|            21411| -73.60540008544922|40.997520446777344|  -73.60081481933594|40.999839782

# IX. Feature Engineering

tiến hành tạo thêm các thuộc tính phục vụ phân tích luồng giao thông

## IX.1. Xử lý dữ liệu thời gian

Chuyển các thuộc tính thời gian về kiểu Timestamp và tạo thêm các thuộc tính:

- `pickup_date`: ngày thực hiện chuyến đi.
- `pickup_hour`: giờ trong ngày.
- `day_of_week`: ngày trong tuần.
- `month`: tháng.
- `time_hour`: mốc thời gian theo từng giờ.

Các thuộc tính này sẽ được sử dụng để phân tích tình trạng giao thông theo thời gian.

In [22]:
from pyspark.sql.functions import (
    col, to_timestamp, to_date,
    hour, dayofweek, month, date_trunc
)

df_feature = df_clean \
    .withColumn(
        "pickup_datetime",
        to_timestamp("pickup_datetime")
    ) \
    .withColumn(
        "dropoff_datetime",
        to_timestamp("dropoff_datetime")
    )

Tạo các thuộc tính thời gian mới từ `pickup_datetime`.

In [23]:
df_feature = df_feature \
    .withColumn(
        "pickup_date",
        to_date("pickup_datetime")
    ) \
    .withColumn(
        "pickup_hour",
        hour("pickup_datetime")
    ) \
    .withColumn(
        "day_of_week",
        dayofweek("pickup_datetime")
    ) \
    .withColumn(
        "month",
        month("pickup_datetime")
    ) \
    .withColumn(
        "time_hour",
        date_trunc("hour", "pickup_datetime")
    )

Hiển thị các thuộc tính thời gian vừa tạo.

In [24]:
df_feature.select(
    "pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "day_of_week",
    "month",
    "time_hour"
).show(10, False)

+-------------------+-----------+-----------+-----------+-----+-------------------+
|pickup_datetime    |pickup_date|pickup_hour|day_of_week|month|time_hour          |
+-------------------+-----------+-----------+-----------+-----+-------------------+
|2016-03-14 17:24:55|2016-03-14 |17         |2          |3    |2016-03-14 17:00:00|
|2016-06-12 00:43:35|2016-06-12 |0          |1          |6    |2016-06-12 00:00:00|
|2016-01-19 11:35:24|2016-01-19 |11         |3          |1    |2016-01-19 11:00:00|
|2016-04-06 19:32:31|2016-04-06 |19         |4          |4    |2016-04-06 19:00:00|
|2016-03-26 13:30:55|2016-03-26 |13         |7          |3    |2016-03-26 13:00:00|
|2016-01-30 22:01:40|2016-01-30 |22         |7          |1    |2016-01-30 22:00:00|
|2016-06-17 22:34:59|2016-06-17 |22         |6          |6    |2016-06-17 22:00:00|
|2016-05-21 07:54:58|2016-05-21 |7          |7          |5    |2016-05-21 07:00:00|
|2016-05-27 23:12:23|2016-05-27 |23         |6          |5    |2016-05-27 23

## IX.2. Tính khoảng cách chuyến đi

Sử dụng tọa độ điểm đón và điểm trả để ước lượng khoảng cách giữa hai vị trí bằng công thức Haversine.

Khoảng cách được tính theo đơn vị kilomet và lưu vào thuộc tính `distance_km`.

In [25]:
from pyspark.sql.functions import radians, sin, cos, asin, sqrt

lat1 = radians(col("pickup_latitude"))
lon1 = radians(col("pickup_longitude"))

lat2 = radians(col("dropoff_latitude"))
lon2 = radians(col("dropoff_longitude"))

a = (
    sin((lat2 - lat1) / 2) ** 2
    +
    cos(lat1) *
    cos(lat2) *
    sin((lon2 - lon1) / 2) ** 2
)

distance = 6371 * 2 * asin(sqrt(a))

df_feature = df_feature.withColumn(
    "distance_km",
    distance
)

Hiển thị khoảng cách của một số chuyến đi.

In [26]:
df_feature.select(
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
    "distance_km"
).show(10)

+------------------+------------------+------------------+------------------+------------------+
|   pickup_latitude|  pickup_longitude|  dropoff_latitude| dropoff_longitude|       distance_km|
+------------------+------------------+------------------+------------------+------------------+
| 40.76793670654297| -73.9821548461914|40.765602111816406|-73.96463012695312|1.4985207796469109|
|40.738563537597656|-73.98041534423828| 40.73115158081055|-73.99948120117188|1.8055071687958897|
|40.763938903808594| -73.9790267944336|40.710086822509766|-74.00533294677734| 6.385098495252496|
|   40.719970703125|-74.01004028320312| 40.70671844482422|-74.01226806640625|1.4854984227709382|
|40.793209075927734|-73.97305297851562| 40.78252029418945| -73.9729232788086|1.1885884593338851|
| 40.74219512939453|-73.98285675048828|40.749183654785156|-73.99208068847656|1.0989424593055537|
| 40.75783920288086| -73.9690170288086| 40.76589584350586|-73.95740509033203|1.3262785770590748|
| 40.79777908325195|-73.969276

## IX.3. Tính tốc độ trung bình ước lượng

Chuyển thời gian chuyến đi từ giây sang giờ và tính tốc độ trung bình của chuyến taxi.

Tốc độ được tính theo đơn vị km/h.

In [27]:
df_feature = df_feature.withColumn(
    "duration_hour",
    col("trip_duration") / 3600
)

Tính tốc độ trung bình ước lượng cho từng chuyến đi.

In [28]:
df_feature = df_feature.withColumn(
    "speed_kmh",
    col("distance_km") / col("duration_hour")
)

Hiển thị khoảng cách, thời gian và tốc độ của một số chuyến đi.

In [29]:
df_feature.select(
    "trip_duration",
    "distance_km",
    "duration_hour",
    "speed_kmh"
).show(10)

+-------------+------------------+-------------------+------------------+
|trip_duration|       distance_km|      duration_hour|         speed_kmh|
+-------------+------------------+-------------------+------------------+
|          455|1.4985207796469109|0.12638888888888888|11.856428146656878|
|          663|1.8055071687958897|0.18416666666666667| 9.803658835090804|
|         2124| 6.385098495252496|               0.59| 10.82220083941101|
|          429|1.4854984227709382|0.11916666666666667|12.465721030245636|
|          435|1.1885884593338851|0.12083333333333333| 9.836594146211462|
|          443|1.0989424593055537|0.12305555555555556| 8.930457908577862|
|          341|1.3262785770590748|0.09472222222222222|14.001767968952109|
|         1551| 5.714980630789905|0.43083333333333335|13.264945371272507|
|          255|1.3103532828841316|0.07083333333333333|18.499105170128917|
|         1225| 5.121161562140774| 0.3402777777777778|15.049944182617784|
+-------------+------------------+----

## IX.4. Kiểm tra khoảng cách và tốc độ

Sau khi tính khoảng cách và tốc độ, tiến hành thống kê để kiểm tra các giá trị bất thường trước khi sử dụng cho phân tích giao thông.

In [30]:
df_feature.select(
    "distance_km",
    "speed_kmh"
).describe().show()

+-------+-----------------+------------------+
|summary|      distance_km|         speed_kmh|
+-------+-----------------+------------------+
|  count|          1447446|           1447446|
|   mean|3.444477869979986|14.399296359327373|
| stddev|3.892673176283667|7.7140994185546585|
|    min|              0.0|               0.0|
|    max|47.76512886563283| 583.7343126289045|
+-------+-----------------+------------------+



Kiểm tra khoảng cách và tốc độ nhỏ nhất, lớn nhất trong tập dữ liệu.

In [31]:
from pyspark.sql.functions import min, max

df_feature.select(
    min("distance_km").alias("distance_min"),
    max("distance_km").alias("distance_max"),
    min("speed_kmh").alias("speed_min"),
    max("speed_kmh").alias("speed_max")
).show()

+------------+-----------------+---------+-----------------+
|distance_min|     distance_max|speed_min|        speed_max|
+------------+-----------------+---------+-----------------+
|         0.0|47.76512886563283|      0.0|583.7343126289045|
+------------+-----------------+---------+-----------------+



## IX.5. Loại bỏ các giá trị không phù hợp

Loại bỏ các chuyến đi có khoảng cách bằng 0 hoặc tốc độ ước lượng quá lớn để hạn chế ảnh hưởng của dữ liệu bất thường đến kết quả phân tích.

In [32]:
df_feature = df_feature.filter(
    (col("distance_km") > 0) &
    (col("speed_kmh") > 0) &
    (col("speed_kmh") <= 120)
)

Kiểm tra số lượng dữ liệu sau khi tạo và làm sạch các thuộc tính mới.

In [33]:
print(
    "Số dòng sau Feature Engineering:",
    df_feature.count()
)

Số dòng sau Feature Engineering: 1443246


## IX.6. Chia khu vực thành các Grid

Để phân tích mật độ giao thông theo khu vực, tọa độ điểm đón được chia thành các ô lưới (Grid).

Các chuyến taxi có vị trí điểm đón gần nhau sẽ được xếp vào cùng một Grid.

Grid được xác định dựa trên `pickup_latitude` và `pickup_longitude`.

In [34]:
from pyspark.sql.functions import floor

df_feature = df_feature \
    .withColumn(
        "grid_lat",
        floor(col("pickup_latitude") * 100)
    ) \
    .withColumn(
        "grid_lon",
        floor(col("pickup_longitude") * 100)
    )

Tạo mã `grid_id` từ vị trí Latitude và Longitude.

In [35]:
from pyspark.sql.functions import concat_ws

df_feature = df_feature.withColumn(
    "grid_id",
    concat_ws(
        "_",
        col("grid_lat"),
        col("grid_lon")
    )
)

Hiển thị vị trí và mã Grid tương ứng của một số chuyến taxi.

In [36]:
df_feature.select(
    "pickup_latitude",
    "pickup_longitude",
    "grid_lat",
    "grid_lon",
    "grid_id"
).show(10)

+------------------+------------------+--------+--------+----------+
|   pickup_latitude|  pickup_longitude|grid_lat|grid_lon|   grid_id|
+------------------+------------------+--------+--------+----------+
| 40.76793670654297| -73.9821548461914|    4076|   -7399|4076_-7399|
|40.738563537597656|-73.98041534423828|    4073|   -7399|4073_-7399|
|40.763938903808594| -73.9790267944336|    4076|   -7398|4076_-7398|
|   40.719970703125|-74.01004028320312|    4071|   -7402|4071_-7402|
|40.793209075927734|-73.97305297851562|    4079|   -7398|4079_-7398|
| 40.74219512939453|-73.98285675048828|    4074|   -7399|4074_-7399|
| 40.75783920288086| -73.9690170288086|    4075|   -7397|4075_-7397|
| 40.79777908325195|-73.96927642822266|    4079|   -7397|4079_-7397|
|40.738399505615234|-73.99948120117188|    4073|   -7400|4073_-7400|
| 40.74433898925781|-73.98104858398438|    4074|   -7399|4074_-7399|
+------------------+------------------+--------+--------+----------+
only showing top 10 rows


## IX.7. Kiểm tra số lượng khu vực

Kiểm tra số lượng Grid được tạo ra từ dữ liệu taxi sau khi phân vùng không gian.

In [37]:
so_grid = df_feature \
    .select("grid_id") \
    .distinct() \
    .count()

print("Số lượng Grid:", so_grid)

Số lượng Grid: 707


Kiểm tra các Grid có nhiều chuyến taxi nhất.

In [38]:
df_feature.groupBy("grid_id") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

+----------+-----+
|   grid_id|count|
+----------+-----+
|4075_-7398|89397|
|4074_-7399|70086|
|4076_-7399|68403|
|4075_-7399|65342|
|4075_-7400|59823|
|4074_-7400|59295|
|4076_-7397|57689|
|4076_-7398|55232|
|4077_-7396|53610|
|4073_-7399|50454|
+----------+-----+
only showing top 10 rows


## IX.8. Kiểm tra kết quả Feature Engineering

Hiển thị các thuộc tính chính sau quá trình tạo đặc trưng.

In [39]:
df_feature.select(
    "id",
    "pickup_datetime",
    "pickup_hour",
    "day_of_week",
    "time_hour",
    "distance_km",
    "speed_kmh",
    "grid_id"
).show(10, False)

+---------+-------------------+-----------+-----------+-------------------+------------------+------------------+----------+
|id       |pickup_datetime    |pickup_hour|day_of_week|time_hour          |distance_km       |speed_kmh         |grid_id   |
+---------+-------------------+-----------+-----------+-------------------+------------------+------------------+----------+
|id2875421|2016-03-14 17:24:55|17         |2          |2016-03-14 17:00:00|1.4985207796469109|11.856428146656878|4076_-7399|
|id2377394|2016-06-12 00:43:35|0          |1          |2016-06-12 00:00:00|1.8055071687958897|9.803658835090804 |4073_-7399|
|id3858529|2016-01-19 11:35:24|11         |3          |2016-01-19 11:00:00|6.385098495252496 |10.82220083941101 |4076_-7398|
|id3504673|2016-04-06 19:32:31|19         |4          |2016-04-06 19:00:00|1.4854984227709382|12.465721030245636|4071_-7402|
|id2181028|2016-03-26 13:30:55|13         |7          |2016-03-26 13:00:00|1.1885884593338851|9.836594146211462 |4079_-7398|


## Kết quả Feature Engineering

Sau quá trình Feature Engineering, tập dữ liệu đã được bổ sung các thuộc tính:

- `pickup_hour`: giờ bắt đầu chuyến đi.
- `day_of_week`: ngày trong tuần.
- `time_hour`: thời gian được gom theo từng giờ.
- `distance_km`: khoảng cách chuyến đi ước lượng.
- `speed_kmh`: tốc độ trung bình ước lượng.
- `grid_id`: khu vực của điểm đón taxi.

Các thuộc tính này sẽ được sử dụng để phân tích mật độ và tốc độ giao thông theo từng khu vực và từng khoảng thời gian.

## X.1. Tổng hợp dữ liệu giao thông theo khu vực và thời gian

Các chuyến taxi được nhóm theo:

- `grid_id`: khu vực.
- `time_hour`: thời gian theo từng giờ.

Tại mỗi khu vực và mỗi giờ, tính:

- `trip_count`: số lượng chuyến taxi.
- `avg_speed`: tốc độ trung bình.
- `avg_duration`: thời gian chuyến đi trung bình.
- `avg_distance`: khoảng cách chuyến đi trung bình.

In [40]:
from pyspark.sql.functions import count, avg

traffic_hourly = df_feature.groupBy(
    "grid_id",
    "time_hour"
).agg(
    count("*").alias("trip_count"),
    avg("speed_kmh").alias("avg_speed"),
    avg("trip_duration").alias("avg_duration"),
    avg("distance_km").alias("avg_distance"),
    avg("pickup_latitude").alias("latitude"),
    avg("pickup_longitude").alias("longitude")
)

Hiển thị một số dữ liệu giao thông sau khi tổng hợp.

In [41]:
traffic_hourly.orderBy(
    "time_hour",
    "grid_id"
).show(20, False)

+----------+-------------------+----------+------------------+------------------+------------------+------------------+------------------+
|grid_id   |time_hour          |trip_count|avg_speed         |avg_duration      |avg_distance      |latitude          |longitude         |
+----------+-------------------+----------+------------------+------------------+------------------+------------------+------------------+
|4064_-7378|2016-01-01 00:00:00|3         |41.20517638172334 |1687.3333333333333|19.624473618219923|40.645643870035805|-73.77670033772786|
|4066_-7398|2016-01-01 00:00:00|1         |31.16336118538517 |1987.0            |17.20044407648898 |40.66684341430664 |-73.97833251953125|
|4067_-7398|2016-01-01 00:00:00|1         |14.91834284444973 |1529.0            |6.336151724767677 |40.67683029174805 |-73.97222137451172|
|4068_-7398|2016-01-01 00:00:00|2         |20.66813526365496 |760.5             |5.640124779953894 |40.6831111907959  |-73.9767951965332 |
|4068_-7399|2016-01-01 00:0

## X.2. Làm tròn các giá trị thống kê

Làm tròn các giá trị tốc độ, thời gian và khoảng cách để thuận tiện khi quan sát và trình bày kết quả.

In [42]:
from pyspark.sql.functions import round

traffic_hourly = traffic_hourly \
    .withColumn(
        "avg_speed",
        round("avg_speed", 2)
    ) \
    .withColumn(
        "avg_duration",
        round("avg_duration", 2)
    ) \
    .withColumn(
        "avg_distance",
        round("avg_distance", 2)
    ) \
    .withColumn(
        "latitude",
        round("latitude", 6)
    ) \
    .withColumn(
        "longitude",
        round("longitude", 6)
    )

Hiển thị lại bảng dữ liệu sau khi làm tròn.

In [43]:
traffic_hourly.show(10, False)

+----------+-------------------+----------+---------+------------+------------+---------+----------+
|grid_id   |time_hour          |trip_count|avg_speed|avg_duration|avg_distance|latitude |longitude |
+----------+-------------------+----------+---------+------------+------------+---------+----------+
|4075_-7398|2016-06-02 23:00:00|25        |12.78    |699.28      |2.63        |40.754971|-73.974994|
|4075_-7397|2016-02-18 09:00:00|15        |12.61    |594.8       |1.75        |40.757157|-73.966053|
|4075_-7398|2016-03-06 11:00:00|31        |16.29    |537.26      |2.41        |40.754189|-73.974695|
|4075_-7398|2016-06-17 11:00:00|40        |11.01    |908.15      |3.09        |40.755058|-73.974853|
|4075_-7398|2016-06-20 14:00:00|25        |9.91     |702.48      |1.77        |40.755055|-73.975898|
|4075_-7399|2016-04-08 00:00:00|28        |15.71    |656.68      |2.89        |40.75601 |-73.985442|
|4073_-7399|2016-02-24 21:00:00|27        |13.77    |732.26      |3.04        |40.736279|-7

## X.3. Kiểm tra dữ liệu giao thông sau khi tổng hợp

Đếm số lượng bản ghi của bảng dữ liệu giao thông theo Grid và từng giờ.

In [44]:
print(
    "Số bản ghi Traffic Flow:",
    traffic_hourly.count()
)

print(
    "Số cột:",
    len(traffic_hourly.columns)
)

Số bản ghi Traffic Flow: 236142
Số cột: 8


## X.4. Phân tích mật độ taxi theo khu vực

Tính tổng số chuyến taxi tại từng Grid để xác định các khu vực có lưu lượng taxi cao.

In [45]:
traffic_by_grid = df_feature.groupBy(
    "grid_id"
).agg(
    count("*").alias("total_trips"),
    avg("speed_kmh").alias("avg_speed")
)

Hiển thị 10 khu vực có số lượng chuyến taxi lớn nhất.

In [46]:
traffic_by_grid.orderBy(
    col("total_trips").desc()
).show(10)

+----------+-----------+------------------+
|   grid_id|total_trips|         avg_speed|
+----------+-----------+------------------+
|4075_-7398|      89397| 12.86128203118562|
|4074_-7399|      70086|13.063835825976572|
|4076_-7399|      68403|13.277291466889478|
|4075_-7399|      65342|13.146234695849268|
|4075_-7400|      59823|12.704963366897632|
|4074_-7400|      59295|12.819520375422355|
|4076_-7397|      57689|13.917024301334495|
|4076_-7398|      55232| 12.54925308877851|
|4077_-7396|      53610|14.779183335448103|
|4073_-7399|      50454|13.660553617666224|
+----------+-----------+------------------+
only showing top 10 rows


## X.5. Phân tích lưu lượng giao thông theo giờ

Phân tích số lượng chuyến taxi và tốc độ trung bình theo từng giờ trong ngày để xác định các khung giờ có lưu lượng giao thông cao.

In [47]:
traffic_by_hour = df_feature.groupBy(
    "pickup_hour"
).agg(
    count("*").alias("total_trips"),
    avg("speed_kmh").alias("avg_speed"),
    avg("trip_duration").alias("avg_duration")
)

Hiển thị kết quả theo thứ tự từ 0 giờ đến 23 giờ.

In [48]:
traffic_by_hour.orderBy(
    "pickup_hour"
).show(24)

+-----------+-----------+------------------+-----------------+
|pickup_hour|total_trips|         avg_speed|     avg_duration|
+-----------+-----------+------------------+-----------------+
|          0|      52595|  17.7248568059171|785.2361821465919|
|          1|      38092|18.486331589788577|744.2111729497008|
|          2|      27586|18.904234893308963| 706.015623867179|
|          3|      20573| 19.97765261019777|708.4118018762456|
|          4|      15483| 22.20366719318042|744.1187108441517|
|          5|      14745|24.496413449879793|722.2313326551373|
|          6|      32847| 20.56254609450719| 677.200870703565|
|          7|      55057|15.779119238455252|763.3190874911455|
|          8|      66423|12.992984340869185|838.7895156798097|
|          9|      67056|12.646819699824496|847.9900530899547|
|         10|      64829|12.791398010736172| 852.507211278903|
|         11|      67793|12.444325282825476|880.6324841797825|
|         12|      71127|12.266621665062733|882.7794227

## X.6. Xác định các khung giờ có lưu lượng cao

Sắp xếp dữ liệu theo số lượng chuyến taxi giảm dần để xác định các khung giờ có lưu lượng taxi lớn nhất.

In [49]:
traffic_by_hour.orderBy(
    col("total_trips").desc()
).show(10)

+-----------+-----------+------------------+-----------------+
|pickup_hour|total_trips|         avg_speed|     avg_duration|
+-----------+-----------+------------------+-----------------+
|         18|      89726|12.566770522626138|866.5614760493056|
|         19|      89466|13.618760669744155|793.8191044642657|
|         21|      83328|15.822851943242584|780.7981350806451|
|         20|      83304|  15.1849269554844|773.6034164025737|
|         22|      79678|16.050131419494598|808.2707773789502|
|         17|      75680|12.509501933897726|936.7503567653278|
|         14|      73481|12.232284055823396|953.5145411739089|
|         12|      71127|12.266621665062733|882.7794227227354|
|         15|      71007| 12.10713996813045|971.3180813159266|
|         13|      70752|12.457219621407042|902.5110950927183|
+-----------+-----------+------------------+-----------------+
only showing top 10 rows


## X.7. Xác định các khung giờ có tốc độ thấp

Sắp xếp tốc độ trung bình theo thứ tự tăng dần để xác định những khung giờ có tốc độ di chuyển thấp.

In [50]:
traffic_by_hour.orderBy(
    col("avg_speed").asc()
).show(10)

+-----------+-----------+------------------+-----------------+
|pickup_hour|total_trips|         avg_speed|     avg_duration|
+-----------+-----------+------------------+-----------------+
|         15|      71007| 12.10713996813045|971.3180813159266|
|         14|      73481|12.232284055823396|953.5145411739089|
|         12|      71127|12.266621665062733|882.7794227227354|
|         11|      67793|12.444325282825476|880.6324841797825|
|         13|      70752|12.457219621407042|902.5110950927183|
|         17|      75680|12.509501933897726|936.7503567653278|
|         18|      89726|12.566770522626138|866.5614760493056|
|         16|      63556|12.584645973177096|971.0162848511549|
|          9|      67056|12.646819699824496|847.9900530899547|
|         10|      64829|12.791398010736172| 852.507211278903|
+-----------+-----------+------------------+-----------------+
only showing top 10 rows


## X.8. Phân tích giao thông theo ngày trong tuần

Phân tích số lượng chuyến taxi và tốc độ trung bình theo từng ngày trong tuần.

In [51]:
traffic_by_day = df_feature.groupBy(
    "day_of_week"
).agg(
    count("*").alias("total_trips"),
    avg("speed_kmh").alias("avg_speed"),
    avg("trip_duration").alias("avg_duration")
)

Hiển thị kết quả phân tích theo ngày trong tuần.

Trong Spark:

- 1: Chủ nhật
- 2: Thứ hai
- 3: Thứ ba
- 4: Thứ tư
- 5: Thứ năm
- 6: Thứ sáu
- 7: Thứ bảy

In [52]:
traffic_by_day.orderBy(
    "day_of_week"
).show()

+-----------+-----------+------------------+-----------------+
|day_of_week|total_trips|         avg_speed|     avg_duration|
+-----------+-----------+------------------+-----------------+
|          1|     193092|16.728752605249824|768.0337559298158|
|          2|     185483|15.100778812215355|814.7607705288356|
|          3|     200650|  13.7703588415981|859.8855220533267|
|          4|     208095| 13.50289551109825|884.0452822028401|
|          5|     216256|13.466256836899737|902.6619700725066|
|          6|     221153|13.767616372201408|  871.82669011951|
|          7|     218517|14.994939819424705|782.9073252881927|
+-----------+-----------+------------------+-----------------+



Tạo tên ngày trong tuần để kết quả dễ quan sát hơn.

In [53]:
from pyspark.sql.functions import when

traffic_by_day = traffic_by_day.withColumn(
    "day_name",
    when(col("day_of_week") == 1, "Sunday")
    .when(col("day_of_week") == 2, "Monday")
    .when(col("day_of_week") == 3, "Tuesday")
    .when(col("day_of_week") == 4, "Wednesday")
    .when(col("day_of_week") == 5, "Thursday")
    .when(col("day_of_week") == 6, "Friday")
    .when(col("day_of_week") == 7, "Saturday")
)

Hiển thị kết quả theo tên ngày trong tuần.

In [54]:
traffic_by_day.select(
    "day_of_week",
    "day_name",
    "total_trips",
    "avg_speed",
    "avg_duration"
).orderBy(
    "day_of_week"
).show()

+-----------+---------+-----------+------------------+-----------------+
|day_of_week| day_name|total_trips|         avg_speed|     avg_duration|
+-----------+---------+-----------+------------------+-----------------+
|          1|   Sunday|     193092|16.728752605249824|768.0337559298158|
|          2|   Monday|     185483|15.100778812215355|814.7607705288356|
|          3|  Tuesday|     200650|  13.7703588415981|859.8855220533267|
|          4|Wednesday|     208095| 13.50289551109825|884.0452822028401|
|          5| Thursday|     216256|13.466256836899737|902.6619700725066|
|          6|   Friday|     221153|13.767616372201408|  871.82669011951|
|          7| Saturday|     218517|14.994939819424705|782.9073252881927|
+-----------+---------+-----------+------------------+-----------------+



## X.10. Xác định điểm nóng giao thông

Xác định các khu vực có số lượng chuyến taxi cao tại từng thời điểm.

Các khu vực có mật độ taxi cao được xem là những khu vực cần được theo dõi để phát hiện nguy cơ ùn tắc.

In [55]:
traffic_hourly.orderBy(
    col("trip_count").desc()
).select(
    "time_hour",
    "grid_id",
    "trip_count",
    "avg_speed",
    "avg_duration"
).show(20, False)

+-------------------+----------+----------+---------+------------+
|time_hour          |grid_id   |trip_count|avg_speed|avg_duration|
+-------------------+----------+----------+---------+------------+
|2016-01-19 19:00:00|4075_-7398|63        |12.72    |761.3       |
|2016-01-13 20:00:00|4075_-7398|63        |14.09    |632.9       |
|2016-03-23 19:00:00|4075_-7398|62        |12.93    |701.76      |
|2016-03-30 19:00:00|4075_-7398|62        |12.15    |842.92      |
|2016-02-10 20:00:00|4075_-7398|60        |14.26    |652.95      |
|2016-01-13 21:00:00|4075_-7398|60        |13.83    |669.28      |
|2016-04-12 18:00:00|4075_-7398|60        |9.66     |777.05      |
|2016-02-17 19:00:00|4075_-7398|58        |12.03    |730.36      |
|2016-02-03 20:00:00|4075_-7398|58        |13.76    |724.83      |
|2016-02-19 19:00:00|4075_-7398|58        |12.15    |767.31      |
|2016-02-24 19:00:00|4075_-7398|58        |10.61    |823.09      |
|2016-02-03 18:00:00|4075_-7398|58        |9.57     |1011.17  

## X.11. Xác định ngưỡng mật độ và tốc độ từ dữ liệu

Sử dụng các phân vị của dữ liệu để xác định ngưỡng tốc độ thấp và mật độ taxi cao.

Cách tiếp cận này giúp hạn chế việc lựa chọn ngưỡng hoàn toàn thủ công.

In [56]:
speed_quantiles = traffic_hourly.approxQuantile(
    "avg_speed",
    [0.25, 0.5],
    0.01
)

trip_quantiles = traffic_hourly.approxQuantile(
    "trip_count",
    [0.5, 0.75],
    0.01
)

speed_p25 = speed_quantiles[0]
speed_p50 = speed_quantiles[1]

trip_p50 = trip_quantiles[0]
trip_p75 = trip_quantiles[1]

print("Speed P25:", speed_p25)
print("Speed P50:", speed_p50)

print("Trip Count P50:", trip_p50)
print("Trip Count P75:", trip_p75)

Speed P25: 11.6
Speed P50: 14.94
Trip Count P50: 3.0
Trip Count P75: 9.0


## X.12. Phân loại tình trạng giao thông

Tình trạng giao thông được xác định dựa trên sự kết hợp giữa:


- Mật độ chuyến taxi.
- Tốc độ di chuyển trung bình.

Phân loại gồm ba mức:

- `LOW`: giao thông bình thường.
- `MEDIUM`: giao thông đông.
- `HIGH`: nguy cơ ùn tắc cao.

In [57]:
traffic_hourly = traffic_hourly.withColumn(
    "traffic_level",

    when(
        (col("avg_speed") <= speed_p25) &
        (col("trip_count") >= trip_p75),
        "HIGH"
    )

    .when(
        (col("avg_speed") <= speed_p50) |
        (col("trip_count") >= trip_p50),
        "MEDIUM"
    )

    .otherwise("LOW")
)

Hiển thị các khu vực có nguy cơ ùn tắc cao.

In [58]:
traffic_hourly.filter(
    col("traffic_level") == "HIGH"
).orderBy(
    col("trip_count").desc()
).show(20, False)

+----------+-------------------+----------+---------+------------+------------+---------+----------+-------------+
|grid_id   |time_hour          |trip_count|avg_speed|avg_duration|avg_distance|latitude |longitude |traffic_level|
+----------+-------------------+----------+---------+------------+------------+---------+----------+-------------+
|4075_-7398|2016-04-12 18:00:00|60        |9.66     |777.05      |2.06        |40.755044|-73.975191|HIGH         |
|4075_-7398|2016-02-24 19:00:00|58        |10.61    |823.09      |2.43        |40.755137|-73.974601|HIGH         |
|4075_-7398|2016-02-03 18:00:00|58        |9.57     |1011.17     |2.72        |40.756075|-73.974492|HIGH         |
|4075_-7398|2016-04-27 20:00:00|57        |11.03    |841.7       |2.39        |40.755216|-73.974769|HIGH         |
|4075_-7398|2016-03-15 18:00:00|56        |10.3     |800.64      |2.25        |40.75487 |-73.975023|HIGH         |
|4075_-7398|2016-04-27 18:00:00|56        |10.39    |947.02      |2.77        |4

## X.14. Thống kê các mức tình trạng giao thông

Đếm số lượng bản ghi thuộc từng mức tình trạng giao thông.

In [59]:
traffic_hourly.groupBy(
    "traffic_level"
).count().show()

+-------------+------+
|traffic_level| count|
+-------------+------+
|         HIGH| 21131|
|          LOW| 62042|
|       MEDIUM|152969|
+-------------+------+



## X.15. Kiểm tra kết quả phân tích Traffic Flow

Hiển thị các thuộc tính chính được sử dụng cho quá trình phân tích và dự báo giao thông.

In [60]:
traffic_hourly.select(
    "time_hour",
    "grid_id",
    "trip_count",
    "avg_speed",
    "avg_duration",
    "avg_distance",
    "latitude",
    "longitude",
    "traffic_level"
).orderBy(
    "time_hour",
    "grid_id"
).show(20, False)

+-------------------+----------+----------+---------+------------+------------+---------+----------+-------------+
|time_hour          |grid_id   |trip_count|avg_speed|avg_duration|avg_distance|latitude |longitude |traffic_level|
+-------------------+----------+----------+---------+------------+------------+---------+----------+-------------+
|2016-01-01 00:00:00|4064_-7378|3         |41.21    |1687.33     |19.62       |40.645644|-73.7767  |MEDIUM       |
|2016-01-01 00:00:00|4066_-7398|1         |31.16    |1987.0      |17.2        |40.666843|-73.978333|LOW          |
|2016-01-01 00:00:00|4067_-7398|1         |14.92    |1529.0      |6.34        |40.67683 |-73.972221|MEDIUM       |
|2016-01-01 00:00:00|4068_-7398|2         |20.67    |760.5       |5.64        |40.683111|-73.976795|LOW          |
|2016-01-01 00:00:00|4068_-7399|1         |12.04    |300.0       |1.0         |40.68692 |-73.984703|MEDIUM       |
|2016-01-01 00:00:00|4068_-7400|2         |13.93    |939.0       |3.66        |4

## Kết quả phân tích luồng giao thông

Sau quá trình phân tích, dữ liệu taxi đã được tổng hợp theo từng khu vực và từng giờ.

Các thuộc tính chính thu được gồm:

- `trip_count`: số lượng chuyến taxi trong khu vực.
- `avg_speed`: tốc độ trung bình.
- `avg_duration`: thời gian chuyến đi trung bình.
- `avg_distance`: khoảng cách trung bình.
- `traffic_level`: mức độ giao thông.

Dựa trên mật độ taxi và tốc độ di chuyển trung bình, hệ thống bước đầu có thể xác định các khu vực có nguy cơ ùn tắc cao.

Bảng `traffic_hourly` sẽ được sử dụng làm dữ liệu đầu vào cho quá trình xây dựng mô hình Machine Learning dự báo tình trạng giao thông trong 1 giờ tiếp theo.

## XI.1. Xác định thời gian của giờ tiếp theo

Từ thuộc tính `time_hour`, tạo thêm thuộc tính `next_time` bằng cách cộng thêm 1 giờ.

In [61]:
from pyspark.sql.functions import expr

traffic_ml = traffic_hourly.withColumn(
    "next_time",
    expr("time_hour + INTERVAL 1 HOUR")
)

Hiển thị thời gian hiện tại và thời gian sau 1 giờ.

In [62]:
traffic_ml.select(
    "grid_id",
    "time_hour",
    "next_time",
    "avg_speed"
).show(10, False)

+----------+-------------------+-------------------+---------+
|grid_id   |time_hour          |next_time          |avg_speed|
+----------+-------------------+-------------------+---------+
|4075_-7398|2016-06-02 23:00:00|2016-06-03 00:00:00|12.78    |
|4075_-7397|2016-02-18 09:00:00|2016-02-18 10:00:00|12.61    |
|4075_-7398|2016-03-06 11:00:00|2016-03-06 12:00:00|16.29    |
|4075_-7398|2016-06-17 11:00:00|2016-06-17 12:00:00|11.01    |
|4075_-7398|2016-06-20 14:00:00|2016-06-20 15:00:00|9.91     |
|4075_-7399|2016-04-08 00:00:00|2016-04-08 01:00:00|15.71    |
|4073_-7399|2016-02-24 21:00:00|2016-02-24 22:00:00|13.77    |
|4072_-7400|2016-02-19 22:00:00|2016-02-19 23:00:00|13.58    |
|4071_-7401|2016-01-16 00:00:00|2016-01-16 01:00:00|17.14    |
|4076_-7399|2016-06-08 22:00:00|2016-06-08 23:00:00|10.52    |
+----------+-------------------+-------------------+---------+
only showing top 10 rows


## XI.2. Tạo giá trị cần dự đoán

Tạo bảng chứa tốc độ trung bình của từng Grid tại từng thời điểm.

Bảng này sẽ được ghép với dữ liệu hiện tại để xác định tốc độ của chính khu vực đó trong 1 giờ tiếp theo.

In [63]:
next_hour_data = traffic_hourly.select(
    col("grid_id"),
    col("time_hour").alias("next_time"),
    col("avg_speed").alias("next_hour_speed")
)

Ghép dữ liệu hiện tại với dữ liệu của đúng 1 giờ tiếp theo theo cùng `grid_id`.

In [64]:
traffic_ml = traffic_ml.join(
    next_hour_data,
    on=["grid_id", "next_time"],
    how="inner"
)

Hiển thị tốc độ hiện tại và tốc độ của 1 giờ tiếp theo.

In [65]:
traffic_ml.select(
    "grid_id",
    "time_hour",
    "next_time",
    "avg_speed",
    "next_hour_speed"
).orderBy(
    "grid_id",
    "time_hour"
).show(20, False)

+----------+-------------------+-------------------+---------+---------------+
|grid_id   |time_hour          |next_time          |avg_speed|next_hour_speed|
+----------+-------------------+-------------------+---------+---------------+
|4062_-7382|2016-02-14 20:00:00|2016-02-14 21:00:00|3.27     |2.3            |
|4064_-7378|2016-01-01 05:00:00|2016-01-01 06:00:00|38.71    |44.92          |
|4064_-7378|2016-01-01 06:00:00|2016-01-01 07:00:00|44.92    |43.91          |
|4064_-7378|2016-01-01 07:00:00|2016-01-01 08:00:00|43.91    |42.58          |
|4064_-7378|2016-01-01 08:00:00|2016-01-01 09:00:00|42.58    |53.65          |
|4064_-7378|2016-01-01 09:00:00|2016-01-01 10:00:00|53.65    |36.8           |
|4064_-7378|2016-01-01 10:00:00|2016-01-01 11:00:00|36.8     |39.46          |
|4064_-7378|2016-01-01 11:00:00|2016-01-01 12:00:00|39.46    |44.81          |
|4064_-7378|2016-01-01 14:00:00|2016-01-01 15:00:00|34.4     |24.86          |
|4064_-7378|2016-01-01 15:00:00|2016-01-01 16:00:00|

## XI.3. Tạo thuộc tính thời gian

Từ `time_hour`, tạo thêm các thuộc tính:

- `hour`: giờ trong ngày.
- `day_of_week`: ngày trong tuần.
- `month`: tháng.

Các thuộc tính này giúp mô hình học được sự khác biệt giữa các khung giờ và ngày khác nhau.

In [66]:
from pyspark.sql.functions import hour, dayofweek, month

traffic_ml = traffic_ml \
    .withColumn(
        "hour",
        hour("time_hour")
    ) \
    .withColumn(
        "day_of_week",
        dayofweek("time_hour")
    ) \
    .withColumn(
        "month",
        month("time_hour")
    )

Kiểm tra các thuộc tính sẽ sử dụng cho Machine Learning.

In [67]:
traffic_ml.select(
    "grid_id",
    "time_hour",
    "hour",
    "day_of_week",
    "month",
    "trip_count",
    "avg_speed",
    "avg_duration",
    "avg_distance",
    "latitude",
    "longitude",
    "next_hour_speed"
).show(10, False)

+----------+-------------------+----+-----------+-----+----------+---------+------------+------------+---------+----------+---------------+
|grid_id   |time_hour          |hour|day_of_week|month|trip_count|avg_speed|avg_duration|avg_distance|latitude |longitude |next_hour_speed|
+----------+-------------------+----+-----------+-----+----------+---------+------------+------------+---------+----------+---------------+
|4064_-7378|2016-01-01 05:00:00|5   |6          |1    |3         |38.71    |1523.0      |16.58       |40.645456|-73.776858|44.92          |
|4064_-7378|2016-01-01 08:00:00|8   |6          |1    |2         |42.58    |1644.0      |19.75       |40.645441|-73.776722|53.65          |
|4064_-7378|2016-01-01 14:00:00|14  |6          |1    |5         |34.4     |1368.6      |13.36       |40.645316|-73.776788|24.86          |
|4064_-7378|2016-01-01 15:00:00|15  |6          |1    |1         |24.86    |3144.0      |21.71       |40.646622|-73.778793|31.9           |
|4064_-7378|2016-01-

## XI.4. Kiểm tra dữ liệu Machine Learning

Kiểm tra số lượng bản ghi và giá trị NULL trước khi xây dựng mô hình.

In [68]:
print(
    "Số bản ghi dùng cho Machine Learning:",
    traffic_ml.count()
)

Số bản ghi dùng cho Machine Learning: 176917


Kiểm tra các giá trị NULL trong các thuộc tính sử dụng cho mô hình.

In [69]:
from pyspark.sql.functions import count, when

ml_columns = [
    "hour",
    "day_of_week",
    "month",
    "trip_count",
    "avg_speed",
    "avg_duration",
    "avg_distance",
    "latitude",
    "longitude",
    "next_hour_speed"
]

traffic_ml.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)

    for c in ml_columns
]).show()

+----+-----------+-----+----------+---------+------------+------------+--------+---------+---------------+
|hour|day_of_week|month|trip_count|avg_speed|avg_duration|avg_distance|latitude|longitude|next_hour_speed|
+----+-----------+-----+----------+---------+------------+------------+--------+---------+---------------+
|   0|          0|    0|         0|        0|           0|           0|       0|        0|              0|
+----+-----------+-----+----------+---------+------------+------------+--------+---------+---------------+



## XI.5. Tạo Vector đặc trưng

Sử dụng `VectorAssembler` để kết hợp các thuộc tính đầu vào thành một vector `features` phục vụ cho các thuật toán Machine Learning.

In [70]:
import sys
import numpy

print(sys.executable)
print(numpy.__version__)

%pip install numpy

c:\Code PyCharms\.venv\Scripts\python.exe
2.5.2
Note: you may need to restart the kernel to use updated packages.


In [71]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "hour",
    "day_of_week",
    "month",
    "trip_count",
    "avg_speed",
    "avg_duration",
    "avg_distance",
    "latitude",
    "longitude"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

ml_data = assembler.transform(
    traffic_ml
)

Hiển thị Vector đặc trưng và giá trị cần dự đoán.

In [72]:
ml_data.select(
    "features",
    "next_hour_speed"
).show(10, False)

+-----------------------------------------------------------+---------------+
|features                                                   |next_hour_speed|
+-----------------------------------------------------------+---------------+
|[5.0,6.0,1.0,3.0,38.71,1523.0,16.58,40.645456,-73.776858]  |44.92          |
|[8.0,6.0,1.0,2.0,42.58,1644.0,19.75,40.645441,-73.776722]  |53.65          |
|[14.0,6.0,1.0,5.0,34.4,1368.6,13.36,40.645316,-73.776788]  |24.86          |
|[15.0,6.0,1.0,1.0,24.86,3144.0,21.71,40.646622,-73.778793] |31.9           |
|[16.0,6.0,1.0,2.0,31.9,2455.5,21.76,40.645384,-73.776855]  |26.92          |
|[17.0,6.0,1.0,2.0,26.92,2327.0,17.56,40.64562,-73.776741]  |32.84          |
|[18.0,6.0,1.0,1.0,32.84,2714.0,24.76,40.646709,-73.778656] |28.09          |
|[23.0,6.0,1.0,3.0,45.46,1564.67,19.45,40.646219,-73.777128]|35.9           |
|[0.0,7.0,1.0,4.0,35.9,1880.25,18.51,40.645246,-73.776838]  |50.37          |
|[5.0,7.0,1.0,3.0,29.28,1688.0,13.9,40.645416,-73.776713]   |30.

## XI.6. Chia dữ liệu Train và Test

Do bài toán có yếu tố thời gian, dữ liệu được chia theo thứ tự thời gian thay vì chia ngẫu nhiên.

Khoảng 80% dữ liệu thời gian đầu được sử dụng để huấn luyện và 20% dữ liệu thời gian sau được sử dụng để kiểm thử.

In [73]:
ml_data = ml_data.withColumn(
    "time_value",
    col("time_hour").cast("long")
)

Xác định mốc thời gian 80% của tập dữ liệu.

In [74]:
cutoff = ml_data.approxQuantile(
    "time_value",
    [0.8],
    0.0
)[0]

print("Mốc chia dữ liệu:", cutoff)

Mốc chia dữ liệu: 1464181200.0


Chia dữ liệu thành tập Train và Test.

In [75]:
train_data = ml_data.filter(
    col("time_value") <= cutoff
)

test_data = ml_data.filter(
    col("time_value") > cutoff
)

Kiểm tra số lượng bản ghi của tập Train và Test.

In [76]:
print(
    "Train:",
    train_data.count()
)

print(
    "Test:",
    test_data.count()
)

Train: 141564
Test: 35353


## XI.7. Mô hình Linear Regression

Xây dựng mô hình Linear Regression để dự đoán `next_hour_speed`.

Mô hình này được sử dụng làm mô hình cơ sở để so sánh với các thuật toán phức tạp hơn.

In [77]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="next_hour_speed"
)

lr_model = lr.fit(
    train_data
)

Sử dụng mô hình Linear Regression để dự đoán trên tập Test.

In [78]:
lr_predictions = lr_model.transform(
    test_data
)

Hiển thị kết quả dự đoán.

In [79]:
lr_predictions.select(
    "grid_id",
    "time_hour",
    "avg_speed",
    "next_hour_speed",
    "prediction"
).show(10, False)

+----------+-------------------+---------+---------------+------------------+
|grid_id   |time_hour          |avg_speed|next_hour_speed|prediction        |
+----------+-------------------+---------+---------------+------------------+
|4064_-7378|2016-05-25 23:00:00|30.95    |54.38          |31.035823788387006|
|4064_-7378|2016-05-26 18:00:00|20.22    |22.13          |23.60995936674226 |
|4064_-7378|2016-05-26 21:00:00|26.94    |25.93          |27.871914906940674|
|4064_-7378|2016-05-27 00:00:00|41.14    |37.25          |34.7817585515113  |
|4064_-7378|2016-05-27 05:00:00|29.18    |23.47          |30.612261924908125|
|4064_-7378|2016-05-27 06:00:00|23.47    |18.6           |25.77260990802597 |
|4064_-7378|2016-05-27 07:00:00|18.6     |17.52          |22.18051621256427 |
|4064_-7378|2016-05-27 18:00:00|18.2     |20.98          |23.993976281234154|
|4064_-7378|2016-05-27 19:00:00|20.98    |28.63          |26.50045451544156 |
|4064_-7378|2016-05-27 20:00:00|28.63    |33.78          |29.595

## XI.8. Đánh giá Linear Regression

Đánh giá mô hình bằng các chỉ số:

- RMSE (Root Mean Squared Error).
- MAE (Mean Absolute Error).
- R² (Coefficient of Determination).

In [80]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol="next_hour_speed",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="next_hour_speed",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="next_hour_speed",
    predictionCol="prediction",
    metricName="r2"
)

In [81]:
lr_rmse = rmse_evaluator.evaluate(
    lr_predictions
)

lr_mae = mae_evaluator.evaluate(
    lr_predictions
)

lr_r2 = r2_evaluator.evaluate(
    lr_predictions
)

print("Linear Regression")
print("RMSE:", lr_rmse)
print("MAE :", lr_mae)
print("R2  :", lr_r2)

Linear Regression
RMSE: 5.25538759354188
MAE : 3.68317942439754
R2  : 0.3775721655230343


## XI.9. Mô hình Random Forest Regressor

Xây dựng mô hình Random Forest Regressor để dự đoán tốc độ giao thông trong 1 giờ tiếp theo.

In [82]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="next_hour_speed",
    numTrees=50,
    seed=42
)

rf_model = rf.fit(
    train_data
)

Dự đoán trên tập dữ liệu Test.

In [83]:
rf_predictions = rf_model.transform(
    test_data
)

Dự đoán trên tập dữ liệu Test.

In [84]:
rf_predictions = rf_model.transform(
    test_data
)

Hiển thị một số kết quả dự đoán của Random Forest.

In [85]:
rf_predictions.select(
    "grid_id",
    "time_hour",
    "avg_speed",
    "next_hour_speed",
    "prediction"
).show(10, False)

+----------+-------------------+---------+---------------+------------------+
|grid_id   |time_hour          |avg_speed|next_hour_speed|prediction        |
+----------+-------------------+---------+---------------+------------------+
|4064_-7378|2016-05-25 23:00:00|30.95    |54.38          |30.79670551959979 |
|4064_-7378|2016-05-26 18:00:00|20.22    |22.13          |24.695211169998498|
|4064_-7378|2016-05-26 21:00:00|26.94    |25.93          |29.48113751531794 |
|4064_-7378|2016-05-27 00:00:00|41.14    |37.25          |34.6612604840946  |
|4064_-7378|2016-05-27 05:00:00|29.18    |23.47          |27.57001308783289 |
|4064_-7378|2016-05-27 06:00:00|23.47    |18.6           |23.983439873202983|
|4064_-7378|2016-05-27 07:00:00|18.6     |17.52          |21.853782340475313|
|4064_-7378|2016-05-27 18:00:00|18.2     |20.98          |24.805996556497934|
|4064_-7378|2016-05-27 19:00:00|20.98    |28.63          |26.77754891927723 |
|4064_-7378|2016-05-27 20:00:00|28.63    |33.78          |31.405

## XI.10. Đánh giá Random Forest

Tính RMSE, MAE và R² của mô hình Random Forest.

In [86]:
rf_rmse = rmse_evaluator.evaluate(
    rf_predictions
)

rf_mae = mae_evaluator.evaluate(
    rf_predictions
)

rf_r2 = r2_evaluator.evaluate(
    rf_predictions
)

print("Random Forest")
print("RMSE:", rf_rmse)
print("MAE :", rf_mae)
print("R2  :", rf_r2)

Random Forest
RMSE: 4.827953515484297
MAE : 3.3892285700643385
R2  : 0.4747021023228958


## XI.11. Mô hình Gradient Boosted Trees

Xây dựng mô hình Gradient Boosted Trees Regressor (GBT) để dự đoán tốc độ giao thông trong 1 giờ tiếp theo.

In [87]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="next_hour_speed",
    maxIter=30,
    seed=42
)

gbt_model = gbt.fit(
    train_data
)

Dự đoán tốc độ giao thông trên tập Test bằng mô hình GBT.

In [88]:
gbt_predictions = gbt_model.transform(
    test_data
)

## XI.12. Đánh giá Gradient Boosted Trees

Tính RMSE, MAE và R² của mô hình GBT.

In [89]:
gbt_rmse = rmse_evaluator.evaluate(
    gbt_predictions
)

gbt_mae = mae_evaluator.evaluate(
    gbt_predictions
)

gbt_r2 = r2_evaluator.evaluate(
    gbt_predictions
)

print("GBT")
print("RMSE:", gbt_rmse)
print("MAE :", gbt_mae)
print("R2  :", gbt_r2)

GBT
RMSE: 4.5186567979729455
MAE : 3.0466382721849636
R2  : 0.539851281721514


## XI.13. So sánh kết quả các mô hình

So sánh các mô hình Linear Regression, Random Forest và Gradient Boosted Trees dựa trên RMSE, MAE và R².

In [90]:
results = [
    (
        "Linear Regression",
        lr_rmse,
        lr_mae,
        lr_r2
    ),
    (
        "Random Forest",
        rf_rmse,
        rf_mae,
        rf_r2
    ),
    (
        "GBT",
        gbt_rmse,
        gbt_mae,
        gbt_r2
    )
]

results_df = spark.createDataFrame(
    results,
    [
        "Model",
        "RMSE",
        "MAE",
        "R2"
    ]
)

results_df.show(
    truncate=False
)

+-----------------+------------------+------------------+------------------+
|Model            |RMSE              |MAE               |R2                |
+-----------------+------------------+------------------+------------------+
|Linear Regression|5.25538759354188  |3.68317942439754  |0.3775721655230343|
|Random Forest    |4.827953515484297 |3.3892285700643385|0.4747021023228958|
|GBT              |4.5186567979729455|3.0466382721849636|0.539851281721514 |
+-----------------+------------------+------------------+------------------+



Sắp xếp các mô hình theo RMSE từ thấp đến cao.

Mô hình có RMSE thấp hơn được xem là mô hình dự đoán tốt hơn trên tập kiểm thử.

In [91]:
results_df.orderBy(
    col("RMSE").asc()
).show(
    truncate=False
)

+-----------------+------------------+------------------+------------------+
|Model            |RMSE              |MAE               |R2                |
+-----------------+------------------+------------------+------------------+
|GBT              |4.5186567979729455|3.0466382721849636|0.539851281721514 |
|Random Forest    |4.827953515484297 |3.3892285700643385|0.4747021023228958|
|Linear Regression|5.25538759354188  |3.68317942439754  |0.3775721655230343|
+-----------------+------------------+------------------+------------------+



## XI.15. Kiểm tra kết quả dự đoán

Hiển thị tốc độ hiện tại, tốc độ thật trong 1 giờ tiếp theo và tốc độ do mô hình GBT dự đoán.

In [92]:
gbt_predictions.select(
    "grid_id",
    "time_hour",
    "avg_speed",
    "next_hour_speed",
    "prediction"
).orderBy(
    "time_hour"
).show(20, False)

+----------+-------------------+---------+---------------+------------------+
|grid_id   |time_hour          |avg_speed|next_hour_speed|prediction        |
+----------+-------------------+---------+---------------+------------------+
|4075_-7397|2016-05-25 21:00:00|14.89    |14.02          |14.916936298771851|
|4078_-7399|2016-05-25 21:00:00|16.99    |12.34          |16.86527461958101 |
|4075_-7398|2016-05-25 21:00:00|14.39    |12.99          |14.808511349510852|
|4070_-7402|2016-05-25 21:00:00|16.28    |10.37          |17.45692410956328 |
|4075_-7399|2016-05-25 21:00:00|13.58    |11.67          |13.828631149475507|
|4071_-7402|2016-05-25 21:00:00|14.37    |14.37          |16.638136310525766|
|4072_-7401|2016-05-25 21:00:00|15.83    |15.99          |14.845134897350137|
|4073_-7399|2016-05-25 21:00:00|13.31    |14.22          |14.24208617690952 |
|4073_-7401|2016-05-25 21:00:00|12.28    |15.62          |13.5514539517672  |
|4064_-7380|2016-05-25 21:00:00|27.9     |29.71          |30.868

## XI.16. Dự đoán mức độ giao thông trong 1 giờ tiếp theo

Chuyển tốc độ dự đoán của mô hình thành các mức tình trạng giao thông:

- `HIGH`: tốc độ dự đoán thấp.
- `MEDIUM`: tốc độ dự đoán trung bình.
- `LOW`: tốc độ dự đoán cao.

In [93]:
prediction_result = gbt_predictions.withColumn(
    "predicted_traffic_level",

    when(
        col("prediction") <= speed_p25,
        "HIGH"
    )

    .when(
        col("prediction") <= speed_p50,
        "MEDIUM"
    )

    .otherwise("LOW")
)

Hiển thị kết quả dự đoán tình trạng giao thông trong 1 giờ tiếp theo.

In [94]:
prediction_result.select(
    "grid_id",
    "time_hour",
    "avg_speed",
    "traffic_level",
    "prediction",
    "predicted_traffic_level"
).show(20, False)

+----------+-------------------+---------+-------------+------------------+-----------------------+
|grid_id   |time_hour          |avg_speed|traffic_level|prediction        |predicted_traffic_level|
+----------+-------------------+---------+-------------+------------------+-----------------------+
|4064_-7378|2016-05-25 23:00:00|30.95    |LOW          |34.39552384894045 |LOW                    |
|4064_-7378|2016-05-26 18:00:00|20.22    |LOW          |23.80913728588508 |LOW                    |
|4064_-7378|2016-05-26 21:00:00|26.94    |LOW          |28.06050301938857 |LOW                    |
|4064_-7378|2016-05-27 00:00:00|41.14    |MEDIUM       |37.802049535133484|LOW                    |
|4064_-7378|2016-05-27 05:00:00|29.18    |LOW          |25.605657308341332|LOW                    |
|4064_-7378|2016-05-27 06:00:00|23.47    |LOW          |20.27855256254698 |LOW                    |
|4064_-7378|2016-05-27 07:00:00|18.6     |LOW          |19.87189793740937 |LOW                    |


## XI.17. Lưu mô hình Machine Learning

Lưu mô hình đã huấn luyện để sử dụng lại trong hệ thống xử lý dữ liệu thời gian thực.

In [97]:
import os

print("HADOOP_HOME =", os.environ.get("HADOOP_HOME"))
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

HADOOP_HOME = C:\hadoop
Hadoop version: 3.5.0


In [98]:
rf_model.write() \
    .overwrite() \
    .save("model/traffic_rf_model")

In [99]:
gbt_model.write() \
    .overwrite() \
    .save("model/traffic_gbt_model")